In [8]:
pip install qiskit==2.3.0 qiskit-aer

Note: you may need to restart the kernel to use updated packages.


In [2]:
from qiskit import *
from qiskit_aer import AerSimulator
import numpy as np

EJERCICIO 1

In [28]:
qc1 = QuantumCircuit(2)

qc1.h(0)
qc1.cx(0,1)
qc1.z(0)
qc1.cx(0,1)
qc1.h(0)

qc1.save_statevector()
qc1.measure_all()
qc1.draw()


┌───┐     ┌───┐     ┌───┐ statevector  ░ ┌─┐   
   q_0: ┤ H ├──■──┤ Z ├──■──┤ H ├──────░───────░─┤M├───
        └───┘┌─┴─┐└───┘┌─┴─┐└───┘      ░       ░ └╥┘┌─┐
   q_1: ─────┤ X ├─────┤ X ├───────────░───────░──╫─┤M├
             └───┘     └───┘           ░       ░  ║ └╥┘
meas: 2/══════════════════════════════════════════╩══╩═
                                                  0  1

In [29]:
sim = AerSimulator()
compiled = transpile(qc1, sim)
result = (sim.run(compiled, shots=10000000)).result()
print(result.get_counts())

{'01': 10000000}


In [30]:
sim = AerSimulator()
compiled = transpile(qc1, sim)
result = sim.run(compiled).result()
sv = result.get_statevector()
print(sv)

Statevector([-6.123234e-17+7.49879891e-33j,  1.000000e+00+0.00000000e+00j,
              0.000000e+00+0.00000000e+00j,  0.000000e+00+0.00000000e+00j],
            dims=(2, 2))


EJERCICIO 2

In [31]:
qc2 = QuantumCircuit(3)

qc2.h(0)
qc2.cx(0,1)
qc2.cx(1,2)

qc2.save_statevector()

qc2.measure_all()
qc2.draw()

┌───┐           statevector  ░ ┌─┐      
   q_0: ┤ H ├──■─────────────░───────░─┤M├──────
        └───┘┌─┴─┐           ░       ░ └╥┘┌─┐   
   q_1: ─────┤ X ├──■────────░───────░──╫─┤M├───
             └───┘┌─┴─┐      ░       ░  ║ └╥┘┌─┐
   q_2: ──────────┤ X ├──────░───────░──╫──╫─┤M├
                  └───┘      ░       ░  ║  ║ └╥┘
meas: 3/════════════════════════════════╩══╩══╩═
                                        0  1  2

In [6]:
sim = AerSimulator()
compiled = transpile(qc2, sim)
result = (sim.run(compiled, shots=10000000)).result()
print(result.get_counts())

{'000': 4999247, '111': 5000753}


In [33]:
sim = AerSimulator()
compiled = transpile(qc2, sim)
result = sim.run(compiled).result()
sv = result.get_statevector()
print(sv)

Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.        +0.j, 0.        +0.j, 0.        +0.j,
             0.        +0.j, 0.70710678+0.j],
            dims=(2, 2, 2))


{'111': 4999103, '000': 5000897} representan aproximadamente 50% cada estado, por lo que se ha generado el estado deseado

In [34]:
qreg = QuantumRegister(4)
creg = ClassicalRegister(4)
qc = QuantumCircuit(qreg,creg)

#Creates the GHZ state
qc.h(1)
qc.cx(1,2)
qc.cx(2,3)

#Teleportation protocol for three qubits (generalization)
qc.cx(0,1)
qc.cx(0,2)

qc.h(0)

qc.save_statevector()

qc.measure([0,1,2],[0,1,2])

#Corrections over the last qubit
with qc.if_test((creg[0], 1)):
        qc.z(3)
with qc.if_test((creg[1], 1)):
        qc.x(3)

qc.draw()


┌───┐ statevector ┌─┐                                »
q9_0: ────────────■────■──┤ H ├──────░──────┤M├────────────────────────────────»
      ┌───┐     ┌─┴─┐  │  └───┘      ░      └╥┘┌─┐                             »
q9_1: ┤ H ├──■──┤ X ├──┼─────────────░───────╫─┤M├─────────────────────────────»
      └───┘┌─┴─┐└───┘┌─┴─┐           ░       ║ └╥┘┌─┐                          »
q9_2: ─────┤ X ├──■──┤ X ├───────────░───────╫──╫─┤M├──────────────────────────»
           └───┘┌─┴─┐└───┘           ░       ║  ║ └╥┘  ┌──────   ┌───┐ ───────┐»
q9_3: ──────────┤ X ├────────────────░───────╫──╫──╫───┤ If-0  ──┤ Z ├  End-0 ├»
                └───┘                ░       ║  ║  ║   └──╥───   └───┘ ───────┘»
                                             ║  ║  ║ ┌────╨─────┐              »
c9: 4/═══════════════════════════════════════╩══╩══╩═╡ c9_0=0x1 ╞══════════════»
                                             0  1  2 └──────────┘              »
«                                  
«q9_0: ────────────────────────────
«                                  
«q9_1: ────────────────────────────
«                                  
«q9_2: ────────────────────────────
«         ┌──────   ┌───┐ ───────┐ 
«q9_3: ───┤ If-0  ──┤ X ├  End-0 ├─
«         └──╥───   └───┘ ───────┘ 
«       ┌────╨─────┐               
«c9: 4/═╡ c9_1=0x1 ╞═══════════════
«       └──────────┘

In [37]:
sim = AerSimulator()
compiled = transpile(qc, sim)
result = sim.run(compiled).result()
sv = result.get_statevector()
print(sv)

Statevector([0.5+0.j, 0.5+0.j, 0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j,
             0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j,
             0.5+0.j, 0.5+0.j],
            dims=(2, 2, 2, 2))


In [35]:
def GHZ(qc,q1,q2,q3):

    #Functions to generate a GHZ state whithin a circuit qc

    qc.h(q1)
    qc.cx(q1,q2)
    qc.cx(q2,q3)

    return qc

In [36]:
def teleport_ghz():
    qreg = QuantumRegister(4)
    creg = ClassicalRegister(4)
    qc = QuantumCircuit(qreg,creg)

    qc = GHZ(qc,1,2,3)

    #Teleportation protocol for three qubits (generalization)
    qc.cx(0,1)
    qc.cx(0,2)

    qc.h(0)

    qc.measure([0,1,2],[0,1,2])

    #Corrections over the last qubit
    with qc.if_test((creg[0], 1)):
            qc.z(3)
    with qc.if_test((creg[1], 1)):
            qc.x(3)

    return qc

In [27]:
qc = teleport_ghz()
qc.draw()

┌───┐   ┌─┐                                       »
q8_0: ────────────■────■──┤ H ├───┤M├───────────────────────────────────────»
      ┌───┐     ┌─┴─┐  │  └┬─┬┘   └╥┘                                       »
q8_1: ┤ H ├──■──┤ X ├──┼───┤M├─────╫────────────────────────────────────────»
      └───┘┌─┴─┐└───┘┌─┴─┐ └╥┘ ┌─┐ ║                                        »
q8_2: ─────┤ X ├──■──┤ X ├──╫──┤M├─╫────────────────────────────────────────»
           └───┘┌─┴─┐└───┘  ║  └╥┘ ║   ┌──────   ┌───┐ ───────┐   ┌──────   »
q8_3: ──────────┤ X ├───────╫───╫──╫───┤ If-0  ──┤ Z ├  End-0 ├───┤ If-0  ──»
                └───┘       ║   ║  ║   └──╥───   └───┘ ───────┘   └──╥───   »
                            ║   ║  ║ ┌────╨─────┐               ┌────╨─────┐»
c8: 4/══════════════════════╩═══╩══╩═╡ c8_0=0x1 ╞═══════════════╡ c8_1=0x1 ╞»
                            1   2  0 └──────────┘               └──────────┘»
«                     
«q8_0: ───────────────
«                     
«q8_1: ───────────────
«                     
«q8_2: ───────────────
«      ┌───┐ ───────┐ 
«q8_3: ┤ X ├  End-0 ├─
«      └───┘ ───────┘ 
«c8: 4/═══════════════
«

EJERCICIO 3